<a href="https://colab.research.google.com/github/zahra-rahman03/Audio-Dialogue-Summarizer/blob/main/Audio_Dialogue_Summarizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# End-to-End ASR & Abstractive Dialogue Summarization Pipeline

In [ ]:
## 1. Setup & Environment Dependencies

In [ ]:

!pip install -q "transformers<5" accelerate gradio librosa torch

ERROR: Operation cancelled by user


In [ ]:
## 2. Model Initialization (Whisper-Small + BART-Large-CNN)

In [ ]:
import torch
from transformers import pipeline

# Check for GPU and set device
if not torch.cuda.is_available():
    print("❌ WARNING: GPU is NOT connected! Please change your runtime to T4 GPU.")
    device = "cpu"
    torch_dtype = torch.float32
else:
    print("✅ GPU Connected!")
    device = "cuda:0"
    torch_dtype = torch.float16

print(f"Loading models into {device}...")

# 1. Load Whisper for Audio Transcription
transcriber = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    torch_dtype=torch_dtype,
    device=device,
)

# 2. Load BART for Summarization
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    torch_dtype=torch_dtype,
    device=device,
)

print("✅ Both models loaded successfully!")

✅ GPU Connected!
Loading models into cuda:0...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:0


model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


✅ Both models loaded successfully!


In [ ]:
## 3. Sliding-Window Audio & Text Processing Pipeline

In [ ]:
def process_audio(audio_path):
    if not audio_path:
        return "No audio provided.", "No summary."

    print("Transcribing audio...")
    # chunk_length_s=30 slices long audio into manageable pieces
    # batch_size=8 processes multiple slices in parallel on the GPU
    transcription_result = transcriber(
        audio_path,
        chunk_length_s=30,
        batch_size=8,
        return_timestamps=True
    )
    full_text = transcription_result["text"]

    print("Generating summary...")
    # The summarizer has a strict token limit.
    # We split the transcript into ~400-word chunks, summarize each, and combine them.
    words = full_text.split()
    chunk_size = 400
    chunks = [' '.join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

    summary_pieces = []
    for chunk in chunks:
        if len(chunk.split()) < 20:
            continue # Skip tiny leftover chunks at the end

        # Distill the chunk into a condensed format
        summary = summarizer(chunk, max_length=130, min_length=30, do_sample=False)
        summary_pieces.append("• " + summary[0]['summary_text'])

    final_summary = "\n\n".join(summary_pieces)

    return full_text.strip(), final_summary

In [ ]:
## 4. Interactive Web Interface (Gradio)

In [ ]:
import gradio as gr

# Build the interactive web interface
demo = gr.Interface(
    fn=process_audio,
    inputs=gr.Audio(type="filepath", label="Upload Lecture/Meeting Audio (.wav, .mp3)"),
    outputs=[
        gr.Textbox(label="Raw Transcript", lines=10),
        gr.Textbox(label="Meeting Summary & Key Points", lines=10)
    ],
    title="🎙️ AI Meeting & Lecture Summarizer",
    description="Upload an audio file. The pipeline uses Whisper for chunked ASR transcription and BART for extractive summarization.",
    flagging_mode="never"  # Updated for Gradio 5+
)

# share=True creates a temporary public link for your CV or portfolio
demo.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6c7149d167d3c8a12f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://6c7149d167d3c8a12f.gradio.live


In [ ]:
## 5. Quantitative Evaluation & ROUGE Benchmarking

In [ ]:
!pip install -q datasets evaluate rouge_score


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00


In [ ]:
import torch
from datasets import load_dataset
from transformers import pipeline
import evaluate
from tqdm.auto import tqdm

print("Loading the SAMSum Dialogue Dataset...")
dataset = load_dataset("knkarthick/samsum", split="test[:50]")

# Initialize the ROUGE evaluation metric
rouge = evaluate.load("rouge")

# Check for GPU
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

# Load the summarizer
print("Loading BART model for evaluation...")
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    device=device,
    torch_dtype=torch_dtype
)

# Convert Arrow Column objects to native Python lists
dialogues = list(dataset["dialogue"])
human_references = list(dataset["summary"])

print(f"Generating AI summaries for {len(dialogues)} dialogues...")
generated_summaries = []
for out in tqdm(summarizer(dialogues, max_length=130, min_length=30, truncation=True, batch_size=8)):
    generated_summaries.append(out["summary_text"])

print("Calculating ROUGE metrics...")
results = rouge.compute(predictions=generated_summaries, references=human_references)

print("\n Model Evaluation Benchmarks (ROUGE Scores):")
print("-" * 50)
print(f"ROUGE-1 (Unigram/Keyword Match):      {results['rouge1'] * 100:.2f}%")
print(f"ROUGE-2 (Bigram/Phrase Match):        {results['rouge2'] * 100:.2f}%")
print(f"ROUGE-L (Longest Common Subsequence): {results['rougeL'] * 100:.2f}%")

Loading the SAMSum Dialogue Dataset...
Loading BART model for evaluation...


Device set to use cuda:0
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Generating AI summaries for 50 dialogues...


  0%|          | 0/50 [00:00<?, ?it/s]

Calculating ROUGE metrics...

 Model Evaluation Benchmarks (ROUGE Scores):
--------------------------------------------------
ROUGE-1 (Unigram/Keyword Match):      32.26%
ROUGE-2 (Bigram/Phrase Match):        9.82%
ROUGE-L (Longest Common Subsequence): 24.95%
